# Skadi BI — Bronze do App

Este notebook extrai do PostgreSQL os dados usados pelos dashboards do Gestor e do Funcionário e grava um snapshot das fontes em tabelas Delta na camada Bronze. A Bronze é compartilhada pelos dois dashboards; a separação por público será feita nas camadas seguintes.

## Tabelas incluídas

- **Núcleo dos dashboards:** `tb_cd`, `tb_usuario`, `tb_camara_frigorifica`, `tb_termometro`, `tb_leitura_temperatura`, `tb_alerta` e `tb_atendimento`.
- **Detalhes e regra de prazo:** `tb_justificativa`, `tb_categoria`, `tb_lote` e `tb_lote_camara_frigorifica`. As três últimas permitem reconstruir a vida útil de referência quando ela não estiver preenchida no alerta.

O notebook não extrai CPF, e-mail, username ou senha. Mantém apenas o nome do usuário, necessário para exibir o responsável no dashboard do Gestor.

## Comportamento da carga

Cada execução lê as tabelas selecionadas da origem e sobrescreve a respectiva tabela Bronze. Assim, a Bronze representa o snapshot atual da origem; esta implementação não é CDC nem preserva versões anteriores de linhas alteradas. A tabela de leituras mantém o histórico que ainda existir em `tb_leitura_temperatura`.

As credenciais devem existir em Databricks Secrets no scope `skadi-bi`, com as chaves `jdbc_url`, `db_username` e `db_password`. Os widgets correspondentes devem conter esses nomes de chave, nunca os valores das credenciais. Ajuste catálogo/schema ao workspace antes da primeira execução.

In [ ]:
import re
from urllib.parse import urlparse

from pyspark.sql import DataFrame
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")

dbutils.widgets.text("secret_scope", "skadi-bi")
dbutils.widgets.text("secret_jdbc_url", "jdbc_url")
dbutils.widgets.text("secret_usuario", "db_username")
dbutils.widgets.text("secret_senha", "db_password")
dbutils.widgets.text("schema_origem", "public")
dbutils.widgets.text("catalogo_destino", "workspace")
dbutils.widgets.text("schema_bronze", "bronze_app")
dbutils.widgets.text("tabela_amostra", "tb_alerta")

SECRET_SCOPE = dbutils.widgets.get("secret_scope").strip()
SECRET_JDBC_URL = dbutils.widgets.get("secret_jdbc_url").strip()
SECRET_USUARIO = dbutils.widgets.get("secret_usuario").strip()
SECRET_SENHA = dbutils.widgets.get("secret_senha").strip()
SCHEMA_ORIGEM = dbutils.widgets.get("schema_origem").strip()
CATALOGO_DESTINO = dbutils.widgets.get("catalogo_destino").strip()
SCHEMA_BRONZE = dbutils.widgets.get("schema_bronze").strip()
TABELA_AMOSTRA = dbutils.widgets.get("tabela_amostra").strip()

def validar_identificador(valor: str, nome: str) -> str:
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", valor):
        raise ValueError(f"Identificador inválido em {nome}: {valor!r}")
    return valor

SCHEMA_ORIGEM = validar_identificador(SCHEMA_ORIGEM, "schema_origem")
CATALOGO_DESTINO = validar_identificador(CATALOGO_DESTINO, "catalogo_destino")
SCHEMA_BRONZE = validar_identificador(SCHEMA_BRONZE, "schema_bronze")

if not all([SECRET_SCOPE, SECRET_JDBC_URL, SECRET_USUARIO, SECRET_SENHA]):
    raise ValueError("Preencha os widgets com o scope e os nomes das chaves do Databricks Secret. Não informe os valores das credenciais nesses widgets.")

JDBC_URL = dbutils.secrets.get(SECRET_SCOPE, SECRET_JDBC_URL).strip()
JDBC_USUARIO = dbutils.secrets.get(SECRET_SCOPE, SECRET_USUARIO)
JDBC_SENHA = dbutils.secrets.get(SECRET_SCOPE, SECRET_SENHA)

if not JDBC_URL.startswith("jdbc:postgresql://"):
    raise ValueError(
        "A chave jdbc_url deve conter uma URL no formato "
        "jdbc:postgresql://HOST:PORTA/BANCO?sslmode=require."
    )

JDBC_PARTES = urlparse(JDBC_URL[len("jdbc:"):])
try:
    JDBC_PORTA = JDBC_PARTES.port
except ValueError as erro:
    raise ValueError("A porta na chave jdbc_url não é válida.") from erro

if not JDBC_PARTES.hostname or not JDBC_PORTA or not JDBC_PARTES.path.strip("/"):
    raise ValueError("A chave jdbc_url precisa conter host, porta e nome do banco.")

DESTINO = f"{CATALOGO_DESTINO}.{SCHEMA_BRONZE}"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DESTINO}")

def ler_postgres(consulta: str) -> DataFrame:
    return (
        spark.read.format("jdbc")
        .option("url", JDBC_URL)
        .option("dbtable", f"({consulta}) AS origem")
        .option("user", JDBC_USUARIO)
        .option("password", JDBC_SENHA)
        .option("driver", "org.postgresql.Driver")
        .option("fetchsize", "10000")
        .load()
    )

def gravar_bronze(df: DataFrame, nome_tabela: str) -> None:
    destino_tabela = f"{DESTINO}.bronze_app_{nome_tabela}"
    (
        df.withColumn("_bronze_ingested_at", F.current_timestamp())
        .write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(destino_tabela)
    )


## Extração e gravação

As consultas selecionam somente os campos necessários. Confirme os nomes de colunas e o schema no banco implantado: a fonte de verdade para esta versão é o DDL PostgreSQL do projeto.

In [ ]:
# Nome lógico da tabela -> colunas necessárias para os dashboards
CONSULTAS = {
    "tb_cd": """
        SELECT id, nome
        FROM {schema}.tb_cd
    """,
    "tb_usuario": """
        SELECT id, nome, nivel_acesso, cod_cd, cod_gestor
        FROM {schema}.tb_usuario
    """,
    "tb_camara_frigorifica": """
        SELECT id, modelo, localizacao, temperatura_min, temperatura_max,
               cod_cd, cod_termometro
        FROM {schema}.tb_camara_frigorifica
    """,
    "tb_termometro": """
        SELECT id, modelo
        FROM {schema}.tb_termometro
    """,
    "tb_leitura_temperatura": """
        SELECT id, cod_termometro, cod_alerta, temperatura, data_hora,
               id_evento_redis
        FROM {schema}.tb_leitura_temperatura
    """,
    "tb_alerta": """
        SELECT id, cod_camara_frigorifica, vida_util_referencia_horas,
               nivel_atual, data_hora, tipo, nivel_gravidade, status
        FROM {schema}.tb_alerta
    """,
    "tb_atendimento": """
        SELECT id, cod_alerta, cod_usuario, data_hora_reconhecimento,
               data_hora_resolucao, status
        FROM {schema}.tb_atendimento
    """,
    "tb_justificativa": """
        SELECT id, cod_atendimento, motivo, descricao, data_hora
        FROM {schema}.tb_justificativa
    """,
    "tb_categoria": """
        SELECT id, nome, temperatura_min, temperatura_max, vida_util_horas
        FROM {schema}.tb_categoria
    """,
    "tb_lote": """
        SELECT id, codigo_lote, cod_categoria, data_fabricacao,
               data_validade, status
        FROM {schema}.tb_lote
    """,
    "tb_lote_camara_frigorifica": """
        SELECT id, cod_lote, cod_camara_frigorifica, data_entrada, data_saida
        FROM {schema}.tb_lote_camara_frigorifica
    """,
}

# Todas as leituras são criadas antes de começar a sobrescrever destinos.
fontes = {
    nome: ler_postgres(sql.format(schema=SCHEMA_ORIGEM))
    for nome, sql in CONSULTAS.items()
}

for nome, df in fontes.items():
    gravar_bronze(df, nome)
    print(f"Gravada: {DESTINO}.bronze_app_{nome}")


## Conferência rápida

Selecione uma tabela no widget `tabela_amostra` e execute a célula para inspecionar uma amostra da camada Bronze. Para conferir outra tabela, atualize o widget e execute novamente.

In [ ]:
TABELAS_BRONZE = set(CONSULTAS)
if TABELA_AMOSTRA not in TABELAS_BRONZE:
    raise ValueError(
        f"tabela_amostra deve ser uma destas opções: {', '.join(sorted(TABELAS_BRONZE))}"
    )

tabela_amostra_completa = f"{DESTINO}.bronze_app_{TABELA_AMOSTRA}"
df_amostra = spark.table(tabela_amostra_completa)
print(f"Tabela: {tabela_amostra_completa}")
df_amostra.printSchema()
display(df_amostra.limit(20))
